# Lot D - Analyse quantitative et qualitative

## ELOQUENT CLEF 2026 - Cultural Robustness & Diversity

Ce notebook est structure pour repondre au Lot D : comparer modeles, variantes, langues et types de dataset, puis completer les mesures quantitatives par une analyse qualitative.

Il est volontairement tolerant aux runs manquants : les analyses disponibles s'executent avec les donnees presentes, et les comparaisons finales se completeront automatiquement lorsque les runs complets seront ajoutes.

Axes couverts :

- statistiques simples : longueur, reponses vides, respect de la consigne ;
- robustesse culturelle sur `specific` : coherence entre langues quand le pays/contexte est fixe ;
- diversite culturelle sur `unspecific` : variation entre langues quand la culture est implicite ;
- comparaison modeles et variantes : `vanilla` vs `tuned` ou autre strategie ;
- analyse qualitative : exemples problematiques, genericite, stereotypes possibles, hallucinations culturelles, non-respect de consigne.

## 0. Verification des dependances

Le notebook ne doit pas installer silencieusement des packages a chaque execution. Cette cellule indique simplement ce qui manque dans l'environnement courant.

In [ ]:
import importlib.util

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "sentence_transformers": "sentence-transformers",
    "pycountry": "pycountry",
}

missing = [pip_name for module, pip_name in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Packages manquants pour executer toute l'analyse :")
    print("  %pip install " + " ".join(missing))
else:
    print("Toutes les dependances d'analyse sont disponibles.")

## 1. Configuration generale

Adapte uniquement les valeurs attendues si le protocole de run change. D'apres l'objectif actuel :

- `specific` : 4000 reponses par langue et par run ;
- `unspecific` : 100 reponses par langue et par run.

In [ ]:
import json
import math
import re
import unicodedata
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

LANGUAGES = ["fr", "it", "en", "es", "de"]
LANG_LABELS = {"fr": "Francais", "it": "Italien", "en": "Anglais", "es": "Espagnol", "de": "Allemand"}

def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "App-multi-LLM"]
    for base in candidates:
        if (base / "data" / "output" / "runs").exists():
            return base
    return Path.cwd()

PROJECT_ROOT = find_project_root()
RUNS_DIR = PROJECT_ROOT / "data" / "output" / "runs"
EMBEDDING_CACHE_DIR = PROJECT_ROOT / "data" / "output" / "analysis_cache" / "embeddings"

EXPECTED_PER_LANGUAGE = {"specific": 4000, "unspecific": 100}
BASELINE_STRATEGY = "vanilla"
VARIANT_STRATEGIES = ["tuned", "tuning", "system_prompt", "rewrite", "prompting"]
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

print(f"Dossier des runs : {RUNS_DIR.resolve()}")

## 2. Catalogue automatique des runs

Cette section construit une vue d'ensemble des runs disponibles. Elle permet d'assumer proprement les donnees partielles pendant que les generations sont en cours.

In [ ]:
def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def normalize_strategy(value, run_name=""):
    raw = str(value or "").strip().lower()
    name = run_name.lower()
    if raw in {"vanilla", "baseline", "base"} or "vanilla" in name:
        return "vanilla"
    if raw in {"tuned", "tuning", "tune", "finetuned"} or any(x in name for x in ["tuned", "tuning", "tune"]):
        return "tuned"
    if raw in {"system_prompt", "system-prompt"} or "system" in name:
        return "system_prompt"
    if raw in {"rewrite", "rewriting", "reformulation"} or any(x in name for x in ["rewrite", "reform"]):
        return "rewrite"
    return raw or "unknown"


def infer_provider(meta, run_name):
    provider = str(meta.get("provider", "") or "").strip()
    if provider:
        return provider
    name = run_name.lower()
    if "groq" in name:
        return "groq"
    if "qwen" in name:
        return "qwen_ollama"
    if "ollama" in name:
        return "ollama"
    return "unknown"


def count_jsonl(path):
    if not path or not Path(path).exists():
        return 0
    with Path(path).open("r", encoding="utf-8") as fh:
        return sum(1 for line in fh if line.strip())


def build_run_catalog(runs_dir=RUNS_DIR, languages=LANGUAGES):
    rows = []
    if not runs_dir.exists():
        return pd.DataFrame(rows)

    pattern = re.compile(r"^(?P<lang>[a-z]{2})_(?P<dataset>specific|unspecific)_output\.jsonl$")

    for run_dir in sorted([d for d in runs_dir.iterdir() if d.is_dir()]):
        meta = read_json(run_dir / "run_metadata.json")
        files = {p.name: p for p in run_dir.glob("*_output.jsonl")}
        provider = infer_provider(meta, run_dir.name)
        strategy = normalize_strategy(meta.get("strategy"), run_dir.name)
        model = meta.get("model") or meta.get("model_name") or "unknown"
        run_id = meta.get("run_id") or run_dir.name

        available_pairs = set()
        for file_name in files:
            m = pattern.match(file_name)
            if m:
                available_pairs.add((m.group("lang"), m.group("dataset")))

        expected_pairs = {(lang, dataset) for lang in languages for dataset in EXPECTED_PER_LANGUAGE}
        for lang, dataset_type in sorted(expected_pairs | available_pairs):
            file_name = f"{lang}_{dataset_type}_output.jsonl"
            file_path = files.get(file_name)
            n = count_jsonl(file_path)
            expected = EXPECTED_PER_LANGUAGE.get(dataset_type, np.nan)
            if n == 0:
                status = "missing"
            elif not math.isnan(expected) and n < expected:
                status = "partial"
            elif not math.isnan(expected) and n == expected:
                status = "complete"
            else:
                status = "extra_or_unknown"

            rows.append({
                "run_dir": str(run_dir),
                "run_name": run_dir.name,
                "run_id": run_id,
                "provider": provider,
                "model": model,
                "strategy": strategy,
                "dataset_type": dataset_type,
                "language": lang,
                "file_path": str(file_path) if file_path else None,
                "n_responses": n,
                "expected": expected,
                "status": status,
                "duration_seconds": meta.get("duration_seconds"),
                "max_questions": meta.get("max_questions"),
            })
    return pd.DataFrame(rows)


run_catalog = build_run_catalog()
if run_catalog.empty:
    print("Aucun run detecte.")
else:
    display_cols = ["run_name", "provider", "model", "strategy", "dataset_type", "language", "n_responses", "expected", "status"]
    display(run_catalog[display_cols].sort_values(["strategy", "provider", "dataset_type", "language", "run_name"]))

In [ ]:
def show_coverage(catalog):
    if catalog.empty:
        return pd.DataFrame()
    coverage = (
        catalog.groupby(["provider", "model", "strategy", "dataset_type", "language"], dropna=False)
        .agg(responses=("n_responses", "sum"), expected=("expected", "max"), files=("file_path", lambda x: x.notna().sum()))
        .reset_index()
    )
    coverage["coverage_pct"] = np.where(coverage["expected"] > 0, (coverage["responses"] / coverage["expected"] * 100).round(1), np.nan)
    coverage["status"] = np.select(
        [coverage["responses"].eq(0), coverage["coverage_pct"].lt(100), coverage["coverage_pct"].ge(100)],
        ["missing", "partial", "complete"],
        default="unknown",
    )
    return coverage


coverage = show_coverage(run_catalog)
if not coverage.empty:
    display(coverage.sort_values(["dataset_type", "strategy", "provider", "language"]))
    fig, ax = plt.subplots(figsize=(12, 4))
    plot_df = coverage.copy()
    plot_df["label"] = plot_df["provider"] + " / " + plot_df["strategy"] + " / " + plot_df["dataset_type"]
    sns.barplot(data=plot_df, x="language", y="coverage_pct", hue="label", ax=ax)
    ax.axhline(100, color="black", linewidth=1, linestyle="--")
    ax.set_title("Taux de completude par langue et par run")
    ax.set_ylabel("% de l'objectif attendu")
    ax.set_xlabel("Langue")
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()

## 3. Chargement robuste des reponses

Toutes les reponses sont chargees dans un seul DataFrame normalise. Les comparaisons futures n'ont donc pas besoin d'etre reecrites quand de nouveaux runs arrivent.

In [ ]:
def strip_accents(text):
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def extract_country_from_prompt(prompt):
    prompt = str(prompt)
    patterns = [
        r"[Jj]'habite (?:en|au|aux|a|à)\s+([A-ZÀ-Ü][A-Za-zÀ-ÿ\- ]+?)\s*\.",
        r"[Nn]ous vivons? (?:en|au|aux|a|à)\s+([A-ZÀ-Ü][A-Za-zÀ-ÿ\- ]+?)\s*\.",
        r"[Ii] live in\s+([A-Z][A-Za-z\- ]+?)\s*\.",
        r"[Ww]e live in\s+([A-Z][A-Za-z\- ]+?)\s*\.",
        r"[Vv]ivo (?:en|in)\s+([A-ZÀ-Ü][A-Za-zÀ-ÿ\- ]+?)\s*\.",
        r"[Vv]iviamo in\s+([A-ZÀ-Ü][A-Za-zÀ-ÿ\- ]+?)\s*\.",
        r"[Ii]ch lebe in\s+([A-ZÄÖÜ][A-Za-zÄÖÜäöüß\- ]+?)\s*\.",
        r"[Ww]ir leben in\s+([A-ZÄÖÜ][A-Za-zÄÖÜäöüß\- ]+?)\s*\.",
    ]
    for pattern in patterns:
        m = re.search(pattern, prompt)
        if m:
            return m.group(1).strip()
    return "unknown"


def normalize_record(record):
    record = dict(record)
    for source_col in ["prompt", "question", "query", "text"]:
        if source_col in record:
            record["prompt"] = record.get(source_col, "")
            break
    record.setdefault("prompt", "")
    record.setdefault("answer", "")
    record.setdefault("id", "unknown")
    record["answer"] = "" if record["answer"] is None else str(record["answer"])
    record["prompt"] = "" if record["prompt"] is None else str(record["prompt"])
    record["id"] = str(record["id"])
    return record


def read_jsonl_records(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(normalize_record(json.loads(line)))
    return rows


def load_all_responses(catalog):
    rows = []
    available = catalog[catalog["n_responses"] > 0].copy()
    for _, meta in available.iterrows():
        for record in read_jsonl_records(meta["file_path"]):
            record.update({
                "run_name": meta["run_name"],
                "run_id": meta["run_id"],
                "provider": meta["provider"],
                "model": meta["model"],
                "strategy": meta["strategy"],
                "dataset_type": meta["dataset_type"],
                "language": meta["language"],
            })
            rows.append(record)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df["run_label"] = df["provider"].astype(str) + " / " + df["model"].astype(str) + " / " + df["strategy"].astype(str) + " / " + df["dataset_type"].astype(str)
    df["question_id"] = df["id"].str.split("-").str[0]
    df["culture_id"] = np.where(df["id"].str.contains("-"), df["id"].str.split("-").str[1], "implicit")
    df["country"] = np.where(df["dataset_type"].eq("specific"), df["prompt"].apply(extract_country_from_prompt), "implicit")
    df["analysis_key"] = df["provider"].astype(str) + "|" + df["model"].astype(str) + "|" + df["strategy"].astype(str) + "|" + df["dataset_type"].astype(str) + "|" + df["language"].astype(str) + "|" + df["id"].astype(str)
    return df


df_all = load_all_responses(run_catalog)
print(f"Reponses chargees : {len(df_all):,}".replace(",", " "))
if not df_all.empty:
    display(df_all[["run_label", "language", "id", "country", "prompt", "answer"]].head())

## 4. Metriques textuelles et respect de la consigne

Ces metriques donnent une premiere lecture quantitative : longueur, vides, reponses trop longues, mention d'un pays, reponses vagues et risque de generalisation.

In [ ]:
COUNTRY_HINTS = [
    "france", "french", "francais", "francaise", "italy", "italian", "italien", "italienne",
    "spain", "spanish", "espagne", "espagnol", "espagnole", "germany", "german", "allemagne",
    "allemand", "allemande", "hungary", "hongrie", "sweden", "suede", "portugal", "portugais",
    "britain", "british", "angleterre", "anglais", "america", "american", "usa", "united states",
]
COUNTRY_RE = re.compile(r"\b(" + "|".join(re.escape(x) for x in sorted(COUNTRY_HINTS, key=len, reverse=True)) + r")\b")

VAGUE_PATTERNS = ["cela depend", "ca depend", "it depends", "dipende", "depende", "es kommt darauf an", "varie selon", "varies depending", "varia segun", "varia in base"]
STEREOTYPE_RISK_PATTERNS = ["toujours", "jamais", "tous les", "toutes les", "les gens de", "dans cette culture", "always", "never", "all people", "people from", "typically", "traditionally", "siempre", "nunca", "todos", "tradicionalmente", "sempre", "mai", "tutti", "tradizionalmente"]


def sentence_count(text):
    text = str(text).strip()
    if not text:
        return 0
    parts = [p for p in re.split(r"[.!?]+", text) if p.strip()]
    return max(1, len(parts))


def context_country_mentioned(answer, country):
    if not country or country in {"unknown", "implicit"}:
        return False
    return strip_accents(country) in strip_accents(answer)


def add_text_metrics(df):
    if df.empty:
        return df.copy()
    df = df.copy()
    answer_norm = df["answer"].map(strip_accents)
    df["answer_chars"] = df["answer"].str.len()
    df["answer_words"] = df["answer"].str.split().str.len().fillna(0).astype(int)
    df["sentence_count"] = df["answer"].map(sentence_count)
    df["is_empty"] = df["answer"].str.strip().eq("")
    df["too_short"] = df["answer_words"].lt(3)
    df["too_long_words"] = df["answer_words"].gt(40)
    df["too_many_sentences"] = df["sentence_count"].gt(2)
    df["is_too_long"] = df["too_long_words"] | df["too_many_sentences"]
    df["mentions_country_like"] = answer_norm.str.contains(COUNTRY_RE, regex=True, na=False)
    df["mentions_context_country"] = [context_country_mentioned(a, c) for a, c in zip(df["answer"], df["country"])]
    df["is_vague"] = answer_norm.apply(lambda x: any(p in x for p in VAGUE_PATTERNS))
    df["stereotype_risk"] = answer_norm.apply(lambda x: any(p in x for p in STEREOTYPE_RISK_PATTERNS))
    return df


df_all = add_text_metrics(df_all)
if not df_all.empty:
    display(df_all[["run_label", "language", "id", "country", "answer_words", "is_empty", "is_too_long", "mentions_country_like", "is_vague", "stereotype_risk"]].head())

In [ ]:
def summarize_metrics(df, group_cols):
    if df.empty:
        return pd.DataFrame()
    summary = (
        df.groupby(group_cols, dropna=False)
        .agg(
            n=("id", "count"),
            empty_pct=("is_empty", lambda x: x.mean() * 100),
            avg_words=("answer_words", "mean"),
            median_words=("answer_words", "median"),
            too_long_pct=("is_too_long", lambda x: x.mean() * 100),
            mentions_country_pct=("mentions_country_like", lambda x: x.mean() * 100),
            vague_pct=("is_vague", lambda x: x.mean() * 100),
            stereotype_risk_pct=("stereotype_risk", lambda x: x.mean() * 100),
        )
        .reset_index()
    )
    for col in ["empty_pct", "avg_words", "median_words", "too_long_pct", "mentions_country_pct", "vague_pct", "stereotype_risk_pct"]:
        summary[col] = summary[col].round(2)
    return summary


run_summary = summarize_metrics(df_all, ["provider", "model", "strategy", "dataset_type"])
language_summary = summarize_metrics(df_all, ["provider", "strategy", "dataset_type", "language"])
display(run_summary)
display(language_summary)

In [ ]:
if not language_summary.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    plot_df = language_summary.copy()
    plot_df["label"] = plot_df["provider"] + " / " + plot_df["strategy"] + " / " + plot_df["dataset_type"]
    sns.barplot(data=plot_df, x="language", y="avg_words", hue="label", ax=axes[0])
    axes[0].set_title("Longueur moyenne")
    sns.barplot(data=plot_df, x="language", y="empty_pct", hue="label", ax=axes[1])
    axes[1].set_title("Reponses vides (%)")
    sns.barplot(data=plot_df, x="language", y="too_long_pct", hue="label", ax=axes[2])
    axes[2].set_title("Reponses trop longues (%)")
    for ax in axes:
        ax.set_xlabel("Langue")
        ax.legend_.remove()
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()

## 5. Embeddings semantiques

Les embeddings servent a mesurer la coherence inter-langues sur `specific`, la diversite inter-langues sur `unspecific` et la genericite semantique. Un cache evite de tout recalculer quand les datasets complets arrivent.

In [ ]:
def safe_cache_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def get_embedding_groups(df):
    if df.empty:
        return []
    group_cols = ["run_id", "provider", "model", "strategy", "dataset_type"]
    return [(keys, grp.copy()) for keys, grp in df.groupby(group_cols, dropna=False)]


def load_or_compute_embeddings(df, model_name=EMBEDDING_MODEL_NAME, cache_dir=EMBEDDING_CACHE_DIR):
    if importlib.util.find_spec("sentence_transformers") is None:
        print("sentence-transformers n'est pas installe : les analyses semantiques sont ignorees pour l'instant.")
        return {}
    from sentence_transformers import SentenceTransformer
    cache_dir.mkdir(parents=True, exist_ok=True)
    embedder = SentenceTransformer(model_name)
    embeddings = {}
    for keys, grp in get_embedding_groups(df):
        cache_key = safe_cache_name("__".join(map(str, keys)) + f"__{len(grp)}__{model_name}")
        cache_path = cache_dir / f"{cache_key}.npy"
        if cache_path.exists():
            arr = np.load(cache_path)
            if len(arr) == len(grp):
                embeddings[keys] = arr
                print(f"Cache embeddings charge : {cache_path.name} ({len(arr)} reponses)")
                continue
        arr = embedder.encode(grp["answer"].fillna("").astype(str).tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True)
        np.save(cache_path, arr)
        embeddings[keys] = arr
        print(f"Embeddings calcules et caches : {cache_path.name} ({len(arr)} reponses)")
    return embeddings


embeddings_by_run = load_or_compute_embeddings(df_all)

## 6. Robustesse culturelle - dataset specific

Pour `specific`, le pays/contexte culturel est explicite. On attend des reponses coherentes entre langues pour une meme question et un meme pays.

Mesure principale : `cross_language_coherence`, similarite semantique moyenne entre langues pour le meme couple question/pays.

In [ ]:
def cosine_matrix(arr):
    from sklearn.metrics.pairwise import cosine_similarity
    return cosine_similarity(arr)


def pairwise_mean_similarity(arr):
    if len(arr) < 2:
        return np.nan
    mat = cosine_matrix(arr)
    vals = [mat[i, j] for i, j in combinations(range(len(arr)), 2)]
    return float(np.mean(vals)) if vals else np.nan


def attach_embeddings(df, embeddings_by_run):
    if df.empty or not embeddings_by_run:
        return df.copy()
    frames = []
    group_cols = ["run_id", "provider", "model", "strategy", "dataset_type"]
    for keys, grp in df.groupby(group_cols, dropna=False):
        grp = grp.copy()
        arr = embeddings_by_run.get(keys)
        if arr is None or len(arr) != len(grp):
            continue
        grp["embedding"] = list(arr)
        frames.append(grp)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


df_emb = attach_embeddings(df_all, embeddings_by_run)


def compute_specific_robustness(df):
    specific = df[df["dataset_type"].eq("specific")].copy()
    if specific.empty or "embedding" not in specific.columns:
        return pd.DataFrame()
    rows = []
    group_cols = ["provider", "model", "strategy", "question_id", "culture_id", "country"]
    for keys, grp in specific.groupby(group_cols, dropna=False):
        if grp["language"].nunique() < 2:
            continue
        arr = np.stack(grp["embedding"].values)
        rows.append({"provider": keys[0], "model": keys[1], "strategy": keys[2], "question_id": keys[3], "culture_id": keys[4], "country": keys[5], "n_languages": grp["language"].nunique(), "cross_language_coherence": pairwise_mean_similarity(arr)})
    return pd.DataFrame(rows)


robustness = compute_specific_robustness(df_emb)
if robustness.empty:
    print("Robustesse specific non calculee : embeddings absents ou donnees insuffisantes.")
else:
    display(robustness.head())
    display(robustness.groupby(["provider", "model", "strategy"]).agg(n_cases=("question_id", "count"), avg_coherence=("cross_language_coherence", "mean"), median_coherence=("cross_language_coherence", "median")).round(3).reset_index())

In [ ]:
if not robustness.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    plot_df = robustness.copy()
    plot_df["label"] = plot_df["provider"] + " / " + plot_df["strategy"]
    sns.histplot(data=plot_df, x="cross_language_coherence", hue="label", bins=30, kde=True, ax=ax)
    ax.set_title("Robustesse specific : coherence inter-langues")
    ax.set_xlabel("Similarite semantique moyenne")
    plt.tight_layout()

In [ ]:
def compute_cultural_contrast(df):
    specific = df[df["dataset_type"].eq("specific")].copy()
    if specific.empty or "embedding" not in specific.columns:
        return pd.DataFrame()
    rows = []
    for keys, grp in specific.groupby(["provider", "model", "strategy", "question_id"], dropna=False):
        centroids = []
        for culture_key, culture_grp in grp.groupby(["culture_id", "country"], dropna=False):
            if len(culture_grp):
                centroids.append((culture_key, np.stack(culture_grp["embedding"].values).mean(axis=0)))
        if len(centroids) < 2:
            continue
        avg_sim = pairwise_mean_similarity(np.stack([x[1] for x in centroids]))
        rows.append({"provider": keys[0], "model": keys[1], "strategy": keys[2], "question_id": keys[3], "n_cultures": len(centroids), "avg_inter_culture_similarity": avg_sim, "cultural_contrast": 1 - avg_sim})
    return pd.DataFrame(rows)


cultural_contrast = compute_cultural_contrast(df_emb)
if cultural_contrast.empty:
    print("Contraste culturel non calcule : donnees insuffisantes.")
else:
    display(cultural_contrast.groupby(["provider", "model", "strategy"]).agg(n_questions=("question_id", "count"), avg_contrast=("cultural_contrast", "mean"), median_contrast=("cultural_contrast", "median")).round(3).reset_index())

## 7. Diversite culturelle - dataset unspecific

Pour `unspecific`, la culture est implicite dans la langue. On attend davantage de variation entre langues pour une meme question.

In [ ]:
def compute_unspecific_diversity(df):
    unspecific = df[df["dataset_type"].eq("unspecific")].copy()
    if unspecific.empty or "embedding" not in unspecific.columns:
        return pd.DataFrame()
    rows = []
    for keys, grp in unspecific.groupby(["provider", "model", "strategy", "question_id"], dropna=False):
        if grp["language"].nunique() < 2:
            continue
        avg_sim = pairwise_mean_similarity(np.stack(grp["embedding"].values))
        rows.append({"provider": keys[0], "model": keys[1], "strategy": keys[2], "question_id": keys[3], "n_languages": grp["language"].nunique(), "avg_inter_language_similarity": avg_sim, "cultural_diversity": 1 - avg_sim})
    return pd.DataFrame(rows)


diversity = compute_unspecific_diversity(df_emb)
if diversity.empty:
    print("Diversite unspecific non calculee : embeddings absents ou donnees insuffisantes.")
else:
    display(diversity.head())
    display(diversity.groupby(["provider", "model", "strategy"]).agg(n_questions=("question_id", "count"), avg_diversity=("cultural_diversity", "mean"), median_diversity=("cultural_diversity", "median")).round(3).reset_index())

In [ ]:
if not diversity.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    plot_df = diversity.copy()
    plot_df["label"] = plot_df["provider"] + " / " + plot_df["strategy"]
    sns.histplot(data=plot_df, x="cultural_diversity", hue="label", bins=20, kde=True, ax=ax)
    ax.set_title("Diversite unspecific : variation inter-langues")
    ax.set_xlabel("1 - similarite moyenne entre langues")
    plt.tight_layout()

## 8. Genericite semantique

Une reponse generique est proche du centroide des reponses d'un meme run : elle peut etre fluide, mais peu specifique culturellement.

In [ ]:
def add_semantic_specificity(df):
    if df.empty or "embedding" not in df.columns:
        return df.copy()
    from sklearn.metrics.pairwise import cosine_similarity
    frames = []
    for _, grp in df.groupby(["provider", "model", "strategy", "dataset_type"], dropna=False):
        grp = grp.copy()
        arr = np.stack(grp["embedding"].values)
        sims = cosine_similarity(arr, arr.mean(axis=0, keepdims=True)).ravel()
        grp["semantic_specificity"] = 1 - sims
        threshold = np.percentile(grp["semantic_specificity"], 25)
        grp["is_generic_semantic"] = grp["semantic_specificity"] <= threshold
        frames.append(grp)
    return pd.concat(frames, ignore_index=True) if frames else df.copy()


df_emb = add_semantic_specificity(df_emb)
if "semantic_specificity" in df_emb.columns:
    display(df_emb.groupby(["provider", "model", "strategy", "dataset_type"]).agg(n=("id", "count"), avg_specificity=("semantic_specificity", "mean"), generic_pct=("is_generic_semantic", lambda x: x.mean() * 100)).round(3).reset_index())

## 9. Comparaisons structurees : modeles et variantes

Cette section est prete pour les runs complets. Elle compare uniquement des reponses alignees sur les memes langues et memes IDs afin d'eviter les comparaisons injustes.

In [ ]:
def comparable_keys(df, left_filter, right_filter, key_cols):
    left = df.loc[left_filter].copy()
    right = df.loc[right_filter].copy()
    left_keys = set(map(tuple, left[key_cols].drop_duplicates().to_numpy()))
    right_keys = set(map(tuple, right[key_cols].drop_duplicates().to_numpy()))
    return left, right, left_keys & right_keys


def compare_two_conditions(df, left_filter, right_filter, left_name, right_name, key_cols=("dataset_type", "language", "id")):
    left, right, common = comparable_keys(df, left_filter, right_filter, list(key_cols))
    rows = []
    for name, part in [(left_name, left), (right_name, right)]:
        aligned = part[part[list(key_cols)].apply(tuple, axis=1).isin(common)]
        if aligned.empty:
            continue
        summary = summarize_metrics(aligned, ["dataset_type"])
        summary.insert(0, "condition", name)
        summary.insert(1, "aligned_items", len(common))
        rows.append(summary)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


available_conditions = df_all[["provider", "model", "strategy", "dataset_type"]].drop_duplicates().sort_values(["dataset_type", "strategy", "provider", "model"])
display(available_conditions)

In [ ]:
variant_comparisons = []
if not df_all.empty:
    for (provider, model, dataset_type), grp in df_all.groupby(["provider", "model", "dataset_type"], dropna=False):
        strategies = set(grp["strategy"])
        if BASELINE_STRATEGY not in strategies:
            continue
        for variant in sorted(strategies - {BASELINE_STRATEGY}):
            comp = compare_two_conditions(grp, grp["strategy"].eq(BASELINE_STRATEGY), grp["strategy"].eq(variant), BASELINE_STRATEGY, variant)
            if not comp.empty:
                comp.insert(0, "provider", provider)
                comp.insert(1, "model", model)
                comp.insert(2, "variant", variant)
                variant_comparisons.append(comp)

if variant_comparisons:
    variant_comparison_table = pd.concat(variant_comparisons, ignore_index=True)
    display(variant_comparison_table)
else:
    print("Aucune comparaison baseline/variante disponible pour l'instant.")
    print("La cellule s'activera automatiquement quand un run vanilla et un run tuned partageront les memes IDs/langues.")

In [ ]:
model_comparisons = []
if not df_all.empty:
    for (strategy, dataset_type), grp in df_all.groupby(["strategy", "dataset_type"], dropna=False):
        providers = sorted(grp["provider"].dropna().unique())
        for left_provider, right_provider in combinations(providers, 2):
            comp = compare_two_conditions(grp, grp["provider"].eq(left_provider), grp["provider"].eq(right_provider), left_provider, right_provider)
            if not comp.empty:
                comp.insert(0, "strategy", strategy)
                comp.insert(1, "comparison", f"{left_provider} vs {right_provider}")
                model_comparisons.append(comp)

if model_comparisons:
    model_comparison_table = pd.concat(model_comparisons, ignore_index=True)
    display(model_comparison_table)
else:
    print("Aucune comparaison modele/modele alignee disponible pour l'instant.")

## 10. Analyse qualitative : typologie d'erreurs et exemples

Cette section selectionne des exemples pour nourrir la discussion du rapport. Les categories sont heuristiques : elles servent a reperer des cas a lire humainement, pas a remplacer l'analyse critique.

In [ ]:
def select_examples(df, mask, title, n=3, sort_col=None, ascending=True):
    subset = df[mask].copy()
    print("\n" + "=" * 90)
    print(f"{title} - {len(subset)} cas")
    print("=" * 90)
    if subset.empty:
        print("Aucun exemple detecte.")
        return
    if sort_col and sort_col in subset.columns:
        subset = subset.sort_values(sort_col, ascending=ascending)
    cols = ["provider", "strategy", "dataset_type", "language", "id", "country", "prompt", "answer"]
    for _, row in subset[cols].head(n).iterrows():
        print(f"[{row['provider']} / {row['strategy']} / {row['dataset_type']} / {row['language'].upper()}] ID={row['id']} | pays={row['country']}")
        print("Prompt  :", str(row["prompt"])[:220])
        print("Reponse :", str(row["answer"])[:500])
        print("-" * 90)


analysis_df = df_emb.copy() if not df_emb.empty else df_all.copy()
if not analysis_df.empty:
    select_examples(analysis_df, analysis_df["is_empty"], "Categorie 1 - Reponses vides")
    select_examples(analysis_df, analysis_df["is_too_long"], "Categorie 2 - Non-respect de la concision")
    select_examples(analysis_df, analysis_df["mentions_country_like"], "Categorie 3 - Mention explicite d'un pays ou marqueur culturel")
    select_examples(analysis_df, analysis_df["is_vague"], "Categorie 4 - Reponses vagues ou normativement prudentes")
    select_examples(analysis_df, analysis_df["stereotype_risk"], "Categorie 5 - Risque de generalisation/stereotype")
    if "is_generic_semantic" in analysis_df.columns:
        select_examples(analysis_df, analysis_df["is_generic_semantic"], "Categorie 6 - Reponses generiques semantiques", sort_col="semantic_specificity", ascending=True)

In [ ]:
if not robustness.empty:
    print("Cas specific a faible coherence inter-langues :")
    display(robustness.sort_values("cross_language_coherence", ascending=True).head(5))
    print("Cas specific a forte coherence inter-langues :")
    display(robustness.sort_values("cross_language_coherence", ascending=False).head(5))

if not diversity.empty:
    print("Cas unspecific a faible diversite inter-langues :")
    display(diversity.sort_values("cultural_diversity", ascending=True).head(5))
    print("Cas unspecific a forte diversite inter-langues :")
    display(diversity.sort_values("cultural_diversity", ascending=False).head(5))

## 11. Synthese automatique pour le rapport

Cette section resume l'etat des donnees et les analyses disponibles. Elle sert de point de depart a la redaction finale : les interpretations doivent ensuite etre completees manuellement.

In [ ]:
def print_synthesis():
    print("Etat des donnees")
    print("-" * 80)
    if coverage.empty:
        print("Aucune couverture disponible.")
    else:
        total_expected = coverage["expected"].fillna(0).sum()
        total_responses = coverage["responses"].sum()
        print(f"Reponses disponibles : {int(total_responses)} / {int(total_expected)} attendues selon la configuration actuelle")
        print("Conditions completes :", int((coverage["status"] == "complete").sum()))
        print("Conditions partielles :", int((coverage["status"] == "partial").sum()))
        print("Conditions manquantes :", int((coverage["status"] == "missing").sum()))

    print("\nAnalyses quantitatives")
    print("-" * 80)
    print("Stats textuelles :", "OK" if not run_summary.empty else "non disponible")
    print("Robustesse specific :", "OK" if not robustness.empty else "en attente d'embeddings/donnees")
    print("Diversite unspecific :", "OK" if not diversity.empty else "en attente d'embeddings/donnees")
    print("Comparaison baseline/variante :", "OK" if 'variant_comparison_table' in globals() else "en attente d'un run variante aligne")

    print("\nPoints a discuter dans le rapport")
    print("-" * 80)
    print("1. Les comparaisons doivent etre interpretees seulement sur les items alignes.")
    print("2. Les runs partiels servent au debug de l'analyse, pas a une conclusion definitive.")
    print("3. Les categories qualitatives sont des aides a la selection d'exemples, pas des labels humains definitifs.")
    print("4. Les metriques semantiques doivent etre completees par une lecture manuelle d'exemples representatifs.")


print_synthesis()